# 11 · Joint AHR learning from observed replay
**Status: UNEXECUTED FIRST DRAFT.**

**Purpose:** warm-start the entire model and jointly update its policy, value, dynamics, Queen allocation, Worker/Scout energies and proposal heads. **Inputs:** a named trained supervised checkpoint, frozen observed replay, configured train game families, and matching recovery gates. **Outputs:** resumable joint checkpoints and measured optimization curves. **Expected runtime:** bounded by `train.max_wall_seconds` (default one hour per invocation); convergence and throughput are unknown.

**Checkpoint directory:** `checkpoints/<run_id>/`. **Exact resume:** rerun with the same run ID, warm-start checkpoint, config, notebook sources and replay hashes. The stage restores its own optimizer state; a warm start transfers weights only when no joint checkpoint exists.

**Method:** this first draft uses offline advantage-weighted actor-critic regression. The actor imitates observed legal actions weighted by a bootstrapped advantage. It does not call logged behavior on-policy, infer expert labels from random actions, or claim that a learned world model verifies its own predictions. Add fresh experience through notebook 03 between explicitly versioned runs. Batches contain single observed transitions; recurrent multi-step credit assignment is a later refinement, not a demonstrated capability of this draft.

**Run All:** starts this real bounded stage only after input and continuation checks. No live environment is opened, so optimizer continuation is exact without claiming restoration of hidden game state. This first draft uses one selected GPU per run.

## Load shared training definitions

In [ ]:
from pathlib import Path
import json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'notebooks' / '00_setup_and_hardware.ipynb').exists()), None)
if ROOT is None:
    raise RuntimeError('Open a kernel in /workspace/arc-agi-3-ahrv2 or its notebooks directory.')
_nb = json.loads((ROOT / 'notebooks' / '00_setup_and_hardware.ipynb').read_text())
for _cell in _nb['cells']:
    if _cell['cell_type'] == 'code' and 'ahr-export' in _cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_cell['source']), '00_setup_and_hardware.ipynb', 'exec'), globals())
load_exports([10], globals())


## Joint stage entry point

In [ ]:
def train_joint(cfg, run_id, root=ROOT):
    return train_stage(cfg, run_id, stage='joint', root=root)


## Select the supervised checkpoint and train
Set the exact path and optionally its SHA256 in the shared configuration. The default intentionally has no invented trained checkpoint. After changing stage or training config, rerun the full-model recovery check in notebook 09 for that configuration.

In [ ]:
cfg = load_config()
cfg['train']['stage'] = 'joint'
cfg['train'].setdefault('warmstart_checkpoint', None)
cfg['train'].setdefault('advantage_temperature', 1.0)
cfg['train'].setdefault('gamma', 0.99)
RUN_ID = 'xl-joint-seed42-v1'
problems = training_readiness(cfg, 'joint')
if problems:
    print('JOINT TRAINING NOT STARTED. Resolve these prerequisites:')
    for problem in problems:
        print(' •', problem)
else:
    joint_report = train_joint(cfg, RUN_ID)
    print(joint_report['status'], joint_report['checkpoint_directory'])


## Review the measured components
A falling energy loss alone is not a success criterion. Check policy, value, calibration, geometry and game outcomes together after benchmark execution.

In [ ]:
curve_path = ROOT / 'metrics' / f'{RUN_ID}.jsonl'
if curve_path.exists():
    import matplotlib.pyplot as plt
    rows = [json.loads(line) for line in curve_path.read_text().splitlines() if line.strip()]
    if rows:
        fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
        steps = [r['step'] for r in rows]
        for ax, keys, title in zip(axes, [('actor', 'value'), ('dynamics', 'reward'), ('worker_energy', 'scout_energy')], ['Policy and value', 'Observed dynamics', 'Energy ranking']):
            for key in keys:
                ax.plot(steps, [r[key] for r in rows], label=key)
            ax.set(xlabel='Completed optimizer step', ylabel='Loss', title=title)
            ax.grid(alpha=0.2)
            ax.legend()
        plt.tight_layout()
        plt.show()
else:
    print('No joint-stage results yet.')
